In [0]:
import pyspark.sql.functions as F

# Criar o banco Gold
spark.sql("CREATE DATABASE IF NOT EXISTS visagio_rocket_lab.gold")

#Carregar as tabelas da Silver para os cruzamentos
df_fat_pedidos = spark.table("visagio_rocket_lab.silver.fat_pedidos")
df_clientes = spark.table("visagio_rocket_lab.silver.dim_consumidores")
df_produtos = spark.table("visagio_rocket_lab.silver.dim_produtos")

# Data Mart 1: Top 5 Estados com mais vendas (Faturamento)
df_vendas_estado = df_fat_pedidos.join(df_clientes, "id_consumidor") \
    .groupBy("estado") \
    .agg(F.sum("valor_item").alias("faturamento_total")) \
    .orderBy(F.col("faturamento_total").desc()) \
    .limit(5)

df_vendas_estado.write.mode("overwrite").saveAsTable("visagio_rocket_lab.gold.dm_top_5_estados")

#Data Mart 2: Top 5 Categorias mais vendidas
df_vendas_categoria = df_fat_pedidos.join(df_produtos, "id_produto") \
    .groupBy("nome_categoria") \
    .agg(F.count("id_pedido").alias("quantidade_pedidos")) \
    .orderBy(F.col("quantidade_pedidos").desc()) \
    .limit(5)

df_vendas_categoria.write.mode("overwrite").saveAsTable("visagio_rocket_lab.gold.dm_top_5_categorias")

print("Gold finalizada com sucesso!")

Gold finalizada com sucesso!
